# TP4 : Implémentation d'un système RAG (Retrieval-Augmented Generation)

## Présentation du TP

Ce travaux pratiques (TP) est dédié à la mise en place et à l'évaluation d'une architecture **RAG** (*Retrieval-Augmented Generation*). L'objectif principal est de pallier les limites de connaissances d'un grand modèle de langage (LLM) en lui permettant d'extraire et de s'appuyer sur des informations issues de documents externes spécifiques avant de générer une réponse.

Au cours de ce TP, nous allons aborder les étapes clés suivantes :
* **Le Document Loading & Splitting** : Chargement et découpage des documents en blocs (*chunks*).
* **L'Indexation & Vectorisation** : Transformation des textes en vecteurs et stockage dans une base de données vectorielle.
* **Le Retrieval (Recherche)** : Extraction des segments de texte les plus pertinents vis-à-vis d'une requête utilisateur.
* **La Génération & Évaluation** : Injection du contexte dans le prompt du LLM et analyse de la fidélité de ses réponses.


# 0. Preparation de l'environement 

Dans le terminal (PowerShell) :
```
uv init
uv venv
Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process
.\.venv\Scripts\Activate.ps1
uv add ipykernel ipywidgets langchain langchain-community langchain-text-splitters langchain-groq langchain-huggingface sentence-transformers chromadb pypdf tiktoken python-dotenv requests
```
Puis créer un fichier `.env` contenant `GROQ_API_KEY=...` (clé Groq, jamais poussée sur GitHub).


In [1]:
import json
import tiktoken
#import pandas as pd||
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader 
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv
import os

C:\Users\pc\AppData\Local\Temp\ipykernel_15212\8798840.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader


##  LLM

In [2]:
load_dotenv(override=True)

True

In [3]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

# 1. Loading the pdf, Chunking

Je vais créer un dossier que je vais appeler `PDFs` et à l'intérieur duquel je vais coller ce fichier PDF `attention.pdf` (pour vous, choisissez votre CV par exemple, ou un fichier que vous connaissez déjà).


In [4]:
pdf_file="./pdfs/attention.pdf"

In [5]:
pdf_loader=PyPDFLoader(pdf_file)

Et si vous voulez charger un dossier qui contient plusieurs fichiers, vous utilisez `PyPDFDirectoryLoader(path="./pdfs")`.

On va utiliser le découpage (*split*). Je rappelle donc que l'on utilise `RecursiveCharacterTextSplitter` avec l'encodeur `tiktoken`. Voici le modèle Tiktoken utilisé : la taille de chaque bloc (*chunk*) est de 300 tokens et le chevauchement (*overlap*) est de 20.

Maintenant, je vais charger le modèle `TextSplitter`. Notez qu'ici, il manque une dépendance. Pour effectuer le découpage, le système vous demande d'installer `pypdf`.

Faites donc : `uv add pypdf`.


In [6]:
text_splitter=RecursiveCharacterTextSplitter.from_tiktoken_encoder(encoding_name='o200k_base',chunk_size=300,chunk_overlap=20)

Dans le terminal de projet ajouter ` uv add pypdf `

La méthode `load_and_split` va charger le fichier, puis appliquer cette stratégie de découpage. Chaque bloc (*chunk*) aura une taille de 300 tokens avec un chevauchement (*overlap*) de 20. Si vous souhaitez vérifier le nombre total de blocs générés, vous pouvez utiliser `len(chunks)`.


In [7]:
chunks=pdf_loader.load_and_split(text_splitter)
len(chunks)


43

# 2.vectore Store - ChromaDB et Embedding

 *Importer le model d'embedding* 



In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [9]:
# On repart d'une collection vide (évite les chunks dupliqués si la cellule est relancée)
try:
    Chroma(collection_name="attention_hf", embedding_function=embedding_model,
           persist_directory="./store").delete_collection()
except Exception:
    pass

vectorstore = Chroma.from_documents(
    chunks, embedding_model,
    collection_name="attention_hf",
    persist_directory="./store"
)

C:\Users\pc\AppData\Local\Temp\ipykernel_15212\4052984492.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  Chroma(collection_name="attention_hf", embedding_function=embedding_model,


Ajouter un dossier "store" et utiliser la commande ` uv add chromadb ` pour ajouter *chromadb*

 Je vais vectoriser les chunks et je les stock dans la base de données vectorielle ici **vectorstore**

# 3.  **Trouver les chunks liée à la questions (Retriever)**

In [10]:
#crée le Retriever 
retriever=vectorstore.as_retriever(search_type='similarity',search_kwargs={'k':5})
# k=5. Donc je vais lui demander de me chercher les cinq chunks les plus proches à la question.


On va faire un test. J'ai la question suivante : 
> « How many identical layers does the encoder of the Transformer have? »

La méthode `retriever.invoke()` a recherché les blocs (*chunks*). Je vais maintenant les afficher.


In [11]:
#test
retrieved_chunks=retriever.invoke(" How many identical layers does the encoder of the Transformer have? ")
print(retrieved_chunks)
print("nombre de chunks est :",len(retrieved_chunks))


[Document(metadata={'producer': 'pdfTeX-1.40.25', 'page_label': '3', 'source': './pdfs/attention.pdf', 'subject': '', 'moddate': '2024-04-10T21:11:43+00:00', 'page': 2, 'keywords': '', 'total_pages': 15, 'author': '', 'trapped': '/False', 'title': '', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00'}, page_content='Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-\nwise fully connected feed-forward network. We employ 

Maintenant, pour la partie RAG, on va créer notre prompt système (*system prompt*).


# 4.  *Prompt Design*

## 4.1 Prompt template

In [12]:
prompt_template="""
Answer the following question based only on provided context The context is about the research paper "Attention Is All You Need"
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
{context}
</context>
<question>
{question}
</question>
"""

### Note sur le contexte
Le prompt précise que le contexte concerne l'article *Attention Is All You Need*. Le contexte est délimité par la balise `<context>` et la question par `<question>`.

### Gestion des hallucinations
Si la réponse ne figure pas dans le contexte, le LLM doit répondre : *"Je ne sais pas!"*. Cela limite les hallucinations.


## 4.2 *Retrieving the Relevant Documents*

### Passage à la partie *retrieving*

Voici ma démarche : je récupère les documents, puis je les **convertis**. Pour chaque document (`relevant_document_chunks`), je sélectionne uniquement le contenu de la page (`d.page_content`), ce qui me génère une liste de textes. Enfin, nous allons concaténer cette liste de *chunks* de texte en utilisant le séparateur `". "`.


In [13]:
user_input=" How many identical layers does the encoder of the Transformer have? "

relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)


context_for_query

'Figure 1: The Transformer - model architecture.\nThe Transformer follows this overall architecture using stacked self-attention and point-wise, fully\nconnected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,\nrespectively.\n3.1 Encoder and Decoder Stacks\nEncoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two\nsub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-\nwise fully connected feed-forward network. We employ a residual connection [11] around each of\nthe two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is\nLayerNorm(x + Sublayer(x)), where Sublayer(x) is the function implemented by the sub-layer\nitself. To facilitate these residual connections, all sub-layers in the model, as well as the embedding\nlayers, produce outputs of dimension dmodel = 512.\nDecoder: The decoder is also composed of a stack of N = 6ident

In [14]:
len(relevant_document_chunks)

5

## 4.3 Le prompt

Maintenant, on va injecter dans le prompt le contexte et la question en utilisant template format. Vous voyez donc pour template On a injecté le contexte et la question. Et puis donc le si vous voulez afficher le prompt. Qu'est-ce qu'il contient ? Il contient:

 1. le système message:  ( Answer the following question ............  Je ne sais pas! `` <context> ``).
 2. le contexte:  entre ` <context> ` et `` </context> `` .
 3. question:  entre ` <question> ` et ` </question> ` .




In [15]:
prompt=prompt_template.format(context=context_for_query,question=user_input)
print(prompt)


Answer the following question based only on provided context The context is about the research paper "Attention Is All You Need"
The context is delimited by <context> tag
The user question is delimited by <question> tag
If the answer is not found in the context, answer: Je ne sais pas!
<context>
Figure 1: The Transformer - model architecture.
The Transformer follows this overall architecture using stacked self-attention and point-wise, fully
connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1,
respectively.
3.1 Encoder and Decoder Stacks
Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two
sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-
wise fully connected feed-forward network. We employ a residual connection [11] around each of
the two sub-layers, followed by layer normalization [ 1]. That is, the output of each sub-layer is
LayerNorm(x + Sublayer(

Donc, ça c'est le prompt.

# 5. *Récupérer la réponse de la part de llm*

 Maintenant on va utiliser le prompt. On va poser cette question à LLM.

In [16]:
# langchain-groq est installé via le terminal : uv add langchain-groq

In [17]:
from langchain_groq import ChatGroq
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)
groundness_checker = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [18]:
resp=llm.invoke(prompt)
from IPython.display import Markdown
display(Markdown(resp.content))

The encoder of the Transformer has **6 identical layers**.

## 6. Regroupement dans une application RAG


### Définition de la fonction RAG

Nous allons définir `RAG` comme notre fonction principale de RAG. Elle prendra en paramètres : `query`, `llm` et `prompt_template`.

**Comment fonctionne cette fonction RAG ?**
1. **Le Retriever** : On lui passe la question (`query`), il recherche les documents pertinents et les convertit en une liste de textes. Chaque texte représente le contenu d'une page (`page_content`).
2. **La Concaténation** : Nous allons concaténer cette liste de textes à l'aide d'un séparateur. Ce texte final devient notre **contexte**.
3. **La Génération** : On injecte ce contexte et la question dans le `prompt_template`. On appelle ensuite la méthode `invoke()` et on retourne le contenu généré.

Voici donc la fonction qui nous intéresse. Pour pouvoir la tester, il vous suffira de faire appel à `RAG()` à chaque fois que vous voudrez lui poser une question.



In [19]:
def RAG(query,llm=llm,prompt_template=prompt_template):
    context_docs=retriever.invoke(query)
    context_list=[d.page_content for d in context_docs]
    context_for_query=". ".join(context_list)   
    prompt=prompt_template.format(context=context_for_query, question=query)
    resp=llm.invoke(prompt)
    return resp.content

### Appel de RAG 

In [20]:
resp=RAG(" Quelle est la capitale de la France ? ")
display(Markdown(resp))

Je ne sais pas!

# 7. Évaluation de la réponse du LLM


## 7.1 Architecture

![Logo](./rag_juge.gif)

In [21]:
#question ou query 
user_input=" How many identical layers does the encoder of the Transformer have? "

Récupérer le context 

In [22]:
relevant_document_chunks=retriever.invoke(user_input)
context_list=[d.page_content for d in relevant_document_chunks]
context_for_query=". ".join(context_list)

In [23]:
user_message_template="""
###Question
{question}
###Context
{context}
###Answer
{answer}
"""

In [24]:
# Réponse d'un RAG
answer=RAG(" How many identical layers does the encoder of the Transformer have? ")
display(Markdown(answer))

The encoder of the Transformer has **6 identical layers**.

## 7.2- *Groundness*

Le prompt pour le juge 

In [25]:
groundness_rater_system_message="""
Vous êtes chargé d'évaluer des réponses générées par une IA à des questions posées par des utilisateurs.
On vous présentera une question, le contexte utilisé par le système d'IA pour générer la réponse, ainsi qu'une réponse générée par l'IA à la question.
Dans l'entrée, la question commencera par ###Question, le contexte commencera par ###Context, et la réponse générée par L'IA commencera par ###Answer.
Critères d'évaluation :
La tâche consiste à juger dans quelle mesure la réponse respecte la métrique.

1- La métrique n'est pas respectée du tout
2- La métrique n'est respectée que dans une mesure limitée
3- La métrique est respectée dans une bonne mesure
4- La métrique est respectée en grande partie
5- La métrique est entièrement respectée

Métrique :

La réponse doit être dérivée uniquement des informations présentées dans le contexte.

Instructions:

Écrivez d'abord les étapes nécessaires pour évaluer la réponse selon la métrique.
Donnez une explication étape par étape indiquant si la réponse respecte la métrique, en considérant la question et le contexte comme entrées.
Évaluez ensuite dans quelle mesure la métrique est respectée.
Utilisez les informations précédentes pour noter la réponse selon les critères d'évaluation et attribuer un score.
"""

In [26]:
# Le juge doit être plus puissant que le LLM du RAG
from langchain_groq import ChatGroq
groundness_checker = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

In [27]:
def evaluate(system_message, user_message_template,question, model=groundness_checker):
    retrieved_chunks=retriever.invoke(question)
    context_list=[d.page_content for d in retrieved_chunks]
    context=". ".join(context_list)
    answer=RAG(question) 
    prompt=f"""
     { system_message}\n
     USER:
     {user_message_template.format(question=question, context=context, answer=answer)}
     """
    juge_response=model.invoke(prompt)
    return juge_response.content



In [28]:
#évaluer
resp=evaluate(groundness_rater_system_message, user_message_template, user_input)
display(Markdown(resp))

**Étapes pour évaluer la réponse selon la métrique**

1. **Identifier la question** – Repérer l’information demandée dans la question.  
2. **Extraire les informations pertinentes du contexte** – Lire le texte fourni et relever les passages qui contiennent la réponse à la question.  
3. **Comparer la réponse générée avec le texte du contexte** – Vérifier que chaque élément de la réponse provient exclusivement du contexte et qu’aucune information externe ou inventée n’est ajoutée.  
4. **Déterminer le degré de conformité** – Évaluer dans quelle mesure la réponse se limite aux données du contexte (aucune, partielle, complète).  
5. **Attribuer un score** selon l’échelle de 1 à 5 décrite dans les critères d’évaluation.

---

**Application de ces étapes à l’exemple**

1. **Question** : « How many identical layers does the encoder of the Transformer have? »  
2. **Recherche dans le contexte** :  
   - Le paragraphe « Encoder » indique : *“The encoder is composed of a stack of N = 6 identical layers.”*  
3. **Comparaison** :  
   - La réponse de l’IA est : *“The encoder of the Transformer has **6 identical layers**.”*  
   - Cette phrase reprend exactement l’information du contexte, sans ajouter de détails, de suppositions ou d’informations extérieures.  
4. **Évaluation de la conformité** :  
   - La réponse est entièrement dérivée du texte fourni et ne contient aucun élément qui ne figure pas dans le contexte.  
5. **Score** : La métrique est donc **entièrement respectée**.

---

**Score attribué** : **5** (la métrique est entièrement respectée)

### Échelle de notation pour la fidélité au contexte (Faithfulness)


| Score | Signification | Description |
| :--- | :--- | :--- |
| **5 / 5** | **Excellent (Parfait)** | La réponse utilise **uniquement** les informations du contexte. Aucune hallucination, aucune extrapolation. |
| **4 / 5** | **Bon** | La réponse est correcte et s'appuie sur le contexte, mais la formulation pourrait être légèrement plus précise. |
| **3 / 5** | **Moyen** | La réponse est partiellement vraie, mais elle omet un détail important du contexte ou manque de clarté. |
| **2 / 5** | **Médiocre** | La réponse contient des extrapolations ou des informations non vérifiables avec le contexte fourni. |
| **1 / 5** | **Mauvais (Hallucination)** | La réponse est totalement hors-sujet, fausse, ou invente des données qui ne sont pas du tout dans le contexte. |
